# Task 1: PyTorch Foundations

Welcome to the first task! In this notebook, we'll cover the essential PyTorch concepts you need to implement BERT from scratch.

## What You'll Learn
1. **Tensors** - The fundamental data structure
2. **Autograd** - Automatic differentiation
3. **nn.Module** - Building custom layers
4. **DataLoader** - Efficient batch processing

Let's get started!

## 1. Tensors

In [ ]:
import torch
import numpy as np

# Create tensors from Python lists
scalar = torch.tensor(5)
vector = torch.tensor([1, 2, 3])
matrix = torch.tensor([[1, 2], [3, 4]])

print(f"Scalar: {scalar}, shape: {scalar.shape}")
print(f"Vector: {vector}, shape: {vector.shape}")
print(f"Matrix: {matrix}, shape: {matrix.shape}")

In [ ]:
# Various tensor creation methods
zeros = torch.zeros(3, 4)  # 3x4 matrix of zeros
ones = torch.ones(2, 3, 4)  # 2x3x4 tensor of ones
rand = torch.rand(2, 2)  # Random uniform [0, 1)
randn = torch.randn(3, 3)  # Random normal (Gaussian)

# Create from numpy
np_array = np.array([1, 2, 3])
tensor_from_np = torch.from_numpy(np_array)

# Specify dtype
float_tensor = torch.tensor([1, 2, 3], dtype=torch.float32)
long_tensor = torch.tensor([1.0, 2.0, 3.0], dtype=torch.long)

print(f"Zeros shape: {zeros.shape}")
print(f"From numpy: {tensor_from_np}")
print(f"Float dtype: {float_tensor.dtype}")

In [ ]:
# Common tensor operations
a = torch.tensor([[1, 2], [3, 4]], dtype=torch.float32)
b = torch.tensor([[5, 6], [7, 8]], dtype=torch.float32)

# Element-wise operations
print(f"a + b = {a + b}")
print(f"a * b (element-wise) = {a * b}")
print(f"a @ b (matrix multiply) = {a @ b}")
print(f"torch.matmul(a, b) = {torch.matmul(a, b)}")

# Reshaping
x = torch.randn(2, 3, 4)
print(f"x reshaped: {x.view(2, 12).shape}")
print(f"x transposed: {x.transpose(1, 2).shape}")

## 2. Autograd - Automatic Differentiation

In [ ]:
# Create tensor with requires_grad=True to track operations
x = torch.tensor([1.0, 2.0, 3.0], requires_grad=True)
y = torch.tensor([4.0, 5.0, 6.0], requires_grad=True)

# Define a computation graph
z = x + y
w = z * 2
loss = w.sum()

# Backward pass computes gradients
loss.backward()

print(f"x.grad = {x.grad}")  # dz/dx * dL/dz = 2 * 1 = 2
print(f"y.grad = {y.grad}")

In [ ]:
# Training a simple linear model from scratch
torch.manual_seed(42)

# Fake data: y = 3x + 1
X = torch.tensor([[1], [2], [3], [4], [5]], dtype=torch.float32)
Y = torch.tensor([[4], [7], [10], [13], [16]], dtype=torch.float32)

# Model: y = Wx + b
W = torch.randn(1, 1, requires_grad=True)
b = torch.randn(1, requires_grad=True)

learning_rate = 0.01
epochs = 100

for epoch in range(epochs):
    # Forward pass
    predictions = X @ W + b
    loss = ((predictions - Y) ** 2).mean()
    
    # Backward pass
    loss.backward()
    
    # Update weights (detach from graph first)
    with torch.no_grad():
        W -= learning_rate * W.grad
        b -= learning_rate * b.grad
        W.grad.zero_()
        b.grad.zero_()
    
    if (epoch + 1) % 20 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

print(f"\nLearned W: {W.item():.2f}, b: {b.item():.2f}")
print("(Expected: W ≈ 3.0, b ≈ 1.0)")

## 3. nn.Module - Building Custom Layers

In [ ]:
# Build a custom linear layer from scratch
class LinearLayer(torch.nn.Module):
    def __init__(self, in_features, out_features):
        super().__init__()
        # Initialize weights and bias
        self.weight = torch.nn.Parameter(torch.randn(out_features, in_features))
        self.bias = torch.nn.Parameter(torch.zeros(out_features))
    
    def forward(self, x):
        # y = xW^T + b
        return x @ self.weight.t() + self.bias

# Test the custom layer
linear = LinearLayer(4, 2)
input_tensor = torch.randn(3, 4)  # batch of 3, 4 features
output = linear(input_tensor)
print(f"Input shape: {input_tensor.shape}")
print(f"Output shape: {output.shape}")
print(f"Output:\n{output}")

In [ ]:
# Build a Multi-Layer Perceptron (MLP)
class SimpleMLP(torch.nn.Module):
    def __init__(self, input_size, hidden_size, output_size):
        super().__init__()
        self.layer1 = torch.nn.Linear(input_size, hidden_size)
        self.relu = torch.nn.ReLU()
        self.layer2 = torch.nn.Linear(hidden_size, output_size)
    
    def forward(self, x):
        x = self.layer1(x)
        x = self.relu(x)
        x = self.layer2(x)
        return x

# Test the MLP
mlp = SimpleMLP(input_size=10, hidden_size=20, output_size=5)
input_tensor = torch.randn(4, 10)  # batch of 4
output = mlp(input_tensor)
print(f"MLP Input: {input_tensor.shape}")
print(f"MLP Output: {output.shape}")

# List all parameters
print(f"\nTotal parameters: {sum(p.numel() for p in mlp.parameters())}")

## 4. DataLoader - Efficient Batch Processing

In [ ]:
from torch.utils.data import Dataset, DataLoader

# Custom Dataset
class SimpleDataset(Dataset):
    def __init__(self, size=100):
        # Generate synthetic data: y = 2x + 1 + noise
        self.x = torch.randn(size, 1)
        self.y = 2 * self.x + 1 + torch.randn(size, 1) * 0.1
    
    def __len__(self):
        return len(self.x)
    
    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

# Create dataset and dataloader
dataset = SimpleDataset(size=100)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)

print(f"Dataset size: {len(dataset)}")
print(f"Number of batches: {len(dataloader)}")

# Iterate through batches
for batch_x, batch_y in dataloader:
    print(f"Batch X shape: {batch_x.shape}, Batch Y shape: {batch_y.shape}")
    break  # Just show first batch

In [ ]:
# Complete training loop with DataLoader
model = SimpleMLP(input_size=1, hidden_size=16, output_size=1)
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)
criterion = torch.nn.MSELoss()

epochs = 50
losses = []

for epoch in range(epochs):
    epoch_loss = 0.0
    
    for batch_x, batch_y in dataloader:
        # Forward pass
        predictions = model(batch_x)
        loss = criterion(predictions, batch_y)
        
        # Backward pass
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        epoch_loss += loss.item()
    
    losses.append(epoch_loss / len(dataloader))
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}, Avg Loss: {losses[-1]:.4f}")

print("\nTraining complete!")

## Exercises

Now it's time to practice what you've learned!

### Exercise 1: Tensor Manipulation
Create a 3D tensor of shape (2, 3, 4) filled with random values. Then:
- Reshape it to shape (2, 12)
- Transpose it so the shape becomes (4, 3, 2)
- Slice it to get the first element of the first dimension

In [ ]:
# Your solution here
# TODO: Create a 3D tensor and perform the operations

# Hint: Use torch.rand(), .reshape(), .transpose(), slicing

# SOLUTION:
tensor_3d = torch.rand(2, 3, 4)
reshaped = tensor_3d.reshape(2, 12)
transposed = tensor_3d.transpose(1, 2)
sliced = tensor_3d[0]

print(f"Original: {tensor_3d.shape}")
print(f"Reshaped: {reshaped.shape}")
print(f"Transposed: {transposed.shape}")
print(f"Sliced: {sliced.shape}")

### Exercise 2: Custom Linear Layer
Implement a custom Linear layer with:
- Xavier/Glorot initialization for weights
- Optional bias term
- Test it with a forward pass

In [ ]:
# Your solution here

# SOLUTION:
class XavierLinear(torch.nn.Module):
    def __init__(self, in_features, out_features, bias=True):
        super().__init__()
        self.in_features = in_features
        self.out_features = out_features
        
        # Xavier initialization
        gain = torch.nn.init.calculate_gain('relu')
        self.weight = torch.nn.Parameter(
            torch.randn(out_features, in_features) * gain / np.sqrt(in_features)
        )
        
        if bias:
            self.bias = torch.nn.Parameter(torch.zeros(out_features))
        else:
            self.register_parameter('bias', None)
    
    def forward(self, x):
        return x @ self.weight.t() + self.bias if self.bias is not None else x @ self.weight.t()

# Test
layer = XavierLinear(10, 5)
x = torch.randn(4, 10)
out = layer(x)
print(f"Input: {x.shape} -> Output: {out.shape}")

### Exercise 3: Training Loop
Create a custom Dataset for a classification problem (e.g., binary sentiment: positive/negative). Train a simple MLP to classify the data.

In [ ]:
# Your solution here

# SOLUTION:
class BinaryClassificationDataset(Dataset):
    def __init__(self, num_samples=200, num_features=10):
        # Generate synthetic binary classification data
        self.x = torch.randn(num_samples, num_features)
        # Labels: 1 if sum of features > 0, else 0
        self.y = (self.x.sum(dim=1) > 0).long()
    
    def __len__(self):
        return len(self.x)
    
    def __getitem__(self, idx):
        return self.x[idx], self.y[idx]

# Create data
train_ds = BinaryClassificationDataset()
train_dl = DataLoader(train_ds, batch_size=32, shuffle=True)

# Simple classifier
classifier = torch.nn.Sequential(
    torch.nn.Linear(10, 32),
    torch.nn.ReLU(),
    torch.nn.Linear(32, 2)
)

optimizer = torch.optim.Adam(classifier.parameters(), lr=0.01)
criterion = torch.nn.CrossEntropyLoss()

# Train
for epoch in range(30):
    for x, y in train_dl:
        pred = classifier(x)
        loss = criterion(pred, y)
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
    
    if (epoch + 1) % 10 == 0:
        print(f"Epoch {epoch+1}, Loss: {loss.item():.4f}")

# Evaluate
with torch.no_grad():
    correct = 0
    for x, y in train_dl:
        pred = classifier(x).argmax(dim=1)
        correct += (pred == y).sum().item()
    print(f"Accuracy: {correct / len(train_ds) * 100:.1f}%")

## Summary

In this task, you learned:
- ✅ PyTorch tensors and basic operations
- ✅ Automatic differentiation with autograd
- ✅ Building custom modules with nn.Module
- ✅ Using DataLoader for batch processing
- ✅ Training loops with optimizers

## Next Task
In [Task 2: Self-Attention Mechanism](../task-02-self-attention/overview.html), we'll implement the core attention mechanism that powers BERT!